In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import numpy as np
import re

In [2]:
options = Options()

options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [3]:
# load product href csvs
red_df = pd.read_csv("data/Red_wine_hrefs.csv")
white_df = pd.read_csv("data/White_wine_hrefs.csv")
sparkling_df = pd.read_csv("data/Champagne-Sparkling_wine_hrefs.csv")

# Combine the DataFrames into a single DataFrame
wine_df = pd.concat([red_df, white_df, sparkling_df], ignore_index=True)

# Going to Scrape:
### product description x
### type (of grapes), separated by space x
### vintage x
### state x
### region x
### wine sweetness
### wine body
### winery name x
### product name x
### non-member price x
### member price x

In [4]:
%%time

#i_failure = 475 # i of last failure

hrefs_subset= []
product_names = []
colours = []
product_descriptions = []
types = []
vintages = []
states = []
regions = []
wineries = []
sizes = []
non_member_prices = []
member_prices = []
bodies = []
sweetnesses = []

total_wines = len(wine_df)
starting_i = 0
for i in range(starting_i, len(wine_df), 5):
    
    # get href
    href = wine_df.iloc[i]["href"]
    hrefs_subset.append(href)
    driver.get(href)

    colours.append(wine_df.iloc[i]["type"])
    
    is_loaded = False
    try:
        # expand specifications list
        wait = WebDriverWait(driver, 10)
        expansion_panel = wait.until(EC.element_to_be_clickable((By.XPATH, "//mat-expansion-panel[contains(., 'Product specifications')]")))
        expansion_panel.click()
        is_loaded = True
    except TimeoutException:
        print("!!! no product page for i =", i, "--href", href)

    if is_loaded:    
        # Scrape product name
        product_names.append(driver.find_element(By.CLASS_NAME, "product-title__name").get_attribute("innerText"))
    
    
        # scrape product description
        product_descriptions.append(driver.find_element(By.CLASS_NAME, "product-skeleton__details-description-reg").get_attribute("innerText"))
        
        
        # scrape specifications list
        specification_ul = driver.find_element(By.CLASS_NAME, "product-attribute__items")
        specification_li_elements = specification_ul.find_elements(By.TAG_NAME, "li")
        specifications = {}
        for li in specification_li_elements:
            key = li.find_element(By.CLASS_NAME, "product-attribute__item-key").get_attribute("innerText")
            value = li.find_element(By.CLASS_NAME, "product-attribute__item-value").get_attribute("innerText")
            specifications[key] = value
        
        if "Type" in specifications:
            types.append(specifications["Type"])
        else:
            types.append(np.nan)
        
        if "Current Vintage" in specifications:
            vintages.append(specifications["Current Vintage"])
        else:
            vintages.append(np.nan)
        
        if "State" in specifications:
            states.append(specifications["State"])
        else:
            states.append(np.nan)
        
        if "Region" in specifications:
            regions.append(specifications["Region"])
        else:
            regions.append(np.nan)
        
        if "Wine Maker" in specifications:
            wineries.append(specifications["Wine Maker"])
        else:
            wineries.append(np.nan)
        
        if "Size" in specifications:
            sizes.append(specifications["Size"])
        else:
            sizes.append(np.nan)
    
        if "Wine Sweetness" in specifications:
            sweetnesses.append(specifications["Wine Sweetness"])
        else:
            sweetnesses.append(np.nan)
    
        if "Wine Body" in specifications:
            bodies.append(specifications["Wine Body"])
        else:
            bodies.append(np.nan)
    
    
        # Scrape prices
        pattern = r'\d+\.*\d*' # regex pattern
        price_contect = driver.find_element(By.ID, "mat-tab-content-1-0")
        try:
            non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price-offer").get_attribute("innerText")
            non_member_price = re.search(pattern, non_member_price).group()
            member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
            member_price = re.search(pattern, member_price).group()
        except NoSuchElementException:
            try:
                non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
                non_member_price = re.search(pattern, non_member_price).group()
                member_price = np.nan
                print("no member price")
            except NoSuchElementException:
                non_member_price = np.nan
                member_price = np.nan
                print("no price found for: i =", i, href)
        
        non_member_prices.append(non_member_price)
        member_prices.append(member_price)
        
        # old price scraping - using selected button to find price
        """
        # Scrape prices
        pattern = r'\d+\.*\d*' # regex pattern
        selected_button = driver.find_element(By.XPATH, "//button[contains(@class, 'pack--selected')]")
        try:
            non_member_price = selected_button.find_element(By.CLASS_NAME, "pack__price-offer").get_attribute("innerText")
            non_member_price = re.search(pattern, non_member_price).group()
            member_price = selected_button.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
            member_price = re.search(pattern, member_price).group()
        except NoSuchElementException:
            try:
                non_member_price = selected_button.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
                non_member_price = re.search(pattern, non_member_price).group()
                member_price = np.nan
                print("no member price for:", href)
            except NoSuchElementException:
                non_member_price = np.nan
                member_price = np.nan
                print("!!! no price found for:", href)
        
        non_member_prices.append(non_member_price)
        member_prices.append(member_price)
        """
        
        """
        # Scrape body and sweetness values
        profile_bars = driver.find_elements(By.CLASS_NAME, "body-profile__bar")
        body_bar = profile_bars[0]
        sweetness_bar = profile_bars[1]
        body_indicators = body_bar.find_elements(By.CSS_SELECTOR, 'body-profile-indicator')
        sweetness_indicators = sweetness_bar.find_elements(By.CSS_SELECTOR, 'body-profile-indicator')
        
        def get_indicator_value(indicators):
            for index, indicator in enumerate(indicators):
                opacity = indicator.value_of_css_property('opacity')
                if opacity == '1':
                    index_with_opacity_1 = index
                    return index_with_opacity_1
                else:
                    return np.nan
        
        body = get_indicator_value(body_indicators)
        bodies.append(body)
        sweetness = get_indicator_value(sweetness_indicators)
        sweetnesses.append(sweetness)
        """

        print("scraped i =", i, "of", total_wines-1)

    else:
        product_names.append(np.nan)
        product_descriptions.append(np.nan)
        types.append(np.nan)
        vintages.append(np.nan)
        states.append(np.nan)
        regions.append(np.nan)
        wineries.append(np.nan)
        sizes.append(np.nan)
        non_member_prices.append(np.nan)
        member_prices.append(np.nan)
        bodies.append(np.nan)
        sweetnesses.append(np.nan)

        print("scraped i =", i, "of", total_wines-1)
    

no member price
scraped i = 0 of 6546
scraped i = 5 of 6546
no member price
scraped i = 10 of 6546
no member price
scraped i = 15 of 6546
no member price
scraped i = 20 of 6546
scraped i = 25 of 6546
scraped i = 30 of 6546
no member price
scraped i = 35 of 6546
no member price
scraped i = 40 of 6546
no member price
scraped i = 45 of 6546
no member price
scraped i = 50 of 6546
no member price
scraped i = 55 of 6546
scraped i = 60 of 6546
no member price
scraped i = 65 of 6546
no member price
scraped i = 70 of 6546
no member price
scraped i = 75 of 6546
no member price
scraped i = 80 of 6546
no member price
scraped i = 85 of 6546
no member price
scraped i = 90 of 6546
no member price
scraped i = 95 of 6546
no member price
scraped i = 100 of 6546
no member price
scraped i = 105 of 6546
scraped i = 110 of 6546
no member price
scraped i = 115 of 6546
no member price
scraped i = 120 of 6546
no member price
scraped i = 125 of 6546
no member price
scraped i = 130 of 6546
no member price
scrape

In [5]:
"""
#wine_df = wine_df.iloc[:9]

wine_df["product_name"] = product_names
wine_df["product_description"] = product_descriptions
wine_df["type"] = types
wine_df["vintage"] = vintages
wine_df["state"] = states
wine_df["region"] = regions
wine_df["winery"] = wineries
wine_df["size"] = sizes
wine_df["non_member_price"] = non_member_prices
wine_df["member_price"] = member_prices
wine_df["body"] = bodies
wine_df["sweetness"] = sweetnesses
"""

populated_wine_df = {
    "href" : hrefs_subset,
    "product_name" : product_names,
    "product_description" : product_descriptions,
    "colour" : colours,
    "type" : types,
    "vintage" : vintages,
    "state" : states,
    "region" : regions,
    "winery" : wineries,
    "size" : sizes,
    "non_member_price" : non_member_prices,
    "member_price" : member_prices,
    "body" : bodies,
    "sweetness" : sweetnesses
}

populated_wine_df = pd.DataFrame(populated_wine_df)

In [6]:
populated_wine_df

,href,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness
0,https://www.danmurphys.com.au/product/DM_17275...,Bin 128 Shiraz 2019,Enjoy layers of dark fruit and savoury spice b...,Red,Shiraz,2019,South Australia,Coonawarra,Peter Gago,750mL,50.20,NaN,Full Bodied,Dry
1,https://www.danmurphys.com.au/product/DM_90496...,Penfolds Koonunga Hill Shiraz 2021 750mL,A multi-regional blend of Shiraz from across S...,Red,Shiraz,NaN,South Australia,Multi-Regional Blend,NaN,750mL,12.95,11.65,Full Bodied,Dry
2,https://www.danmurphys.com.au/product/DM_22865...,Estate Shiraz,"At release, this Shiraz is a dark red colour w...",Red,Shiraz,2019,South Australia,Multi-Regional Blend,NaN,750mL,15.95,NaN,Medium-bodied,Dry
3,https://www.danmurphys.com.au/product/DM_54131...,Shiraz,19 Crimes turned criminals into colonists. Upo...,Red,Shiraz,NaN,South Australia,Multi-Regional Blend,NaN,750mL,10.20,NaN,Full Bodied,Light Bodied
4,https://www.danmurphys.com.au/product/DM_72304...,Barossa Cabernet Sauvignon,Cat Amongst The Pigeons has been ruffling a fe...,Red,Cabernet Sauvignon,2022*,South Australia,Barossa Valley,NaN,750mL,14.95,NaN,Full Bodied,Dry
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1305,https://www.danmurphys.com.au/product/DM_50194...,Clos D'Ambonnay Blanc des Noirs 2000,"Mid-yellow colour and a very complex bouquet, ...",Champagne-Sparkling,Pinot Noir,2000,NaN,Champagne,NaN,750mL,3,NaN,NaN,Dry
1306,https://www.danmurphys.com.au/product/DM_23900...,Barossa Valley Stelle Nere Sparkling Shiraz,"Medium depth, crimson with a red rim. Rich, ri...",Champagne-Sparkling,Shiraz,Non Vintage,South Australia,Barossa Valley,Paul Lindner,750mL,55.29,35,Full Bodied,NaN
1307,https://www.danmurphys.com.au/product/DM_20381...,Premium Cuvee NV Sparkling,Brown Brothers Sparkling is wonderfully rich a...,Champagne-Sparkling,Pinot Noir,Non Vintage,Victoria,Multi-Regional Blend,NaN,750mL,21.99,19,Light Bodied,Dry
1308,https://www.danmurphys.com.au/product/DM_15853...,St Aubin,Don’t miss this rare chance to secure a magnif...,Champagne-Sparkling,Chardonnay,NaN,NaN,Champagne,NaN,750mL,129.99,NaN,NaN,Dry


In [7]:
# save df to csv
file_path = f"data/full_wine_dataset.csv"
#populated_wine_df.to_csv(file_path, index=False)
populated_wine_df.to_csv(file_path, index=False)

## Debugging

In [8]:
href

'https://www.danmurphys.com.au/product/DM_107506/perrier-jouet-blanc-de-blanc-champagne?isFromSearch=false&isPersonalised=false&pageName=category_page'

In [9]:
options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


product_descriptions = []
types = []
vintages = []
states = []
regions = []
wineries = []
sizes = []
product_names = []
non_member_prices = []
member_prices = []
bodies = []
sweetnesses = []


# get href
driver.get(href)

is_loaded = False
try:
    # expand specifications list
    wait = WebDriverWait(driver, 10)
    expansion_panel = wait.until(EC.element_to_be_clickable((By.XPATH, "//mat-expansion-panel[contains(., 'Product specifications')]")))
    expansion_panel.click()
    is_loaded = True
except TimeoutException:
    print("!!! no product page for:", href)  

if is_loaded:
    # scrape product description
    product_descriptions.append(driver.find_element(By.CLASS_NAME, "product-skeleton__details-description-reg").get_attribute("innerText"))
    
    
    # scrape specifications list
    specification_ul = driver.find_element(By.CLASS_NAME, "product-attribute__items")
    specification_li_elements = specification_ul.find_elements(By.TAG_NAME, "li")
    specifications = {}
    for li in specification_li_elements:
        key = li.find_element(By.CLASS_NAME, "product-attribute__item-key").get_attribute("innerText")
        value = li.find_element(By.CLASS_NAME, "product-attribute__item-value").get_attribute("innerText")
        specifications[key] = value
    
    if "Type" in specifications:
        types.append(specifications["Type"])
    else:
        types.append(np.nan)
    
    if "Current Vintage" in specifications:
        vintages.append(specifications["Current Vintage"])
    else:
        vintages.append(np.nan)
    
    if "State" in specifications:
        states.append(specifications["State"])
    else:
        states.append(np.nan)
    
    if "Region" in specifications:
        regions.append(specifications["Region"])
    else:
        regions.append(np.nan)
    
    if "Wine Maker" in specifications:
        wineries.append(specifications["Wine Maker"])
    else:
        wineries.append(np.nan)
    
    if "Size" in specifications:
        sizes.append(specifications["Size"])
    else:
        sizes.append(np.nan)
    
    
    # Scrape product name
    product_names.append(driver.find_element(By.CLASS_NAME, "product-title__name").get_attribute("innerText"))
    
    
    # Scrape prices
    pattern = r'\d+\.*\d*' # regex pattern
    price_contect = driver.find_element(By.ID, "mat-tab-content-1-0")
    try:
        non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price-offer").get_attribute("innerText")
        non_member_price = re.search(pattern, non_member_price).group()
        member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
        member_price = re.search(pattern, member_price).group()
    except NoSuchElementException:
        try:
            non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
            non_member_price = re.search(pattern, non_member_price).group()
            member_price = np.nan
            print("no member price")
        except NoSuchElementException:
            non_member_price = np.nan
            member_price = np.nan
            print("no price found for:", href)
    
    non_member_prices.append(non_member_price)
    member_prices.append(member_price)
    
    # old method
    '''
    pattern = r'\d+\.*\d*' # regex pattern
    selected_button = driver.find_element(By.XPATH, "//button[contains(@class, 'pack--selected')]")
    try:
        non_member_price = selected_button.find_element(By.CLASS_NAME, "pack__price-offer").get_attribute("innerText")
        non_member_price = re.search(pattern, non_member_price).group()
        member_price = selected_button.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
        member_price = re.search(pattern, member_price).group()
    except NoSuchElementException:
        try:
            non_member_price = selected_button.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
            non_member_price = re.search(pattern, non_member_price).group()
            member_price = np.nan
            print("no member price")
        except NoSuchElementException:
            non_member_price = np.nan
            member_price = np.nan
            print("no price found for:", href)
    
    non_member_prices.append(non_member_price)
    member_prices.append(member_price)
    '''
    
    
    # Scrape body and sweetness values
    profile_bars = driver.find_elements(By.CLASS_NAME, "body-profile__bar")
    body_bar = profile_bars[0]
    sweetness_bar = profile_bars[1]
    body_indicators = body_bar.find_elements(By.CSS_SELECTOR, 'body-profile-indicator')
    sweetness_indicators = sweetness_bar.find_elements(By.CSS_SELECTOR, 'body-profile-indicator')
    
    def get_indicator_value(indicators):
        for index, indicator in enumerate(indicators):
            opacity = indicator.value_of_css_property('opacity')
            if opacity == '1':
                index_with_opacity_1 = index
                return index_with_opacity_1
            else:
                return np.nan
    
    body = get_indicator_value(body_indicators)
    bodies.append(body)
    sweetness = get_indicator_value(sweetness_indicators)
    sweetnesses.append(sweetness)

else:
    product_names.append(np.nan)
    product_descriptions.append(np.nan)
    types.append(np.nan)
    vintages.append(np.nan)
    states.append(np.nan)
    regions.append(np.nan)
    wineries.append(np.nan)
    sizes.append(np.nan)
    non_member_prices.append(np.nan)
    member_prices.append(np.nan)
    bodies.append(np.nan)
    sweetnesses.append(np.nan)

no member price


IndexError: list index out of range

In [ ]:
#non_member_price

In [ ]:
#re.search(pattern, non_member_price).group()